# P17 — Prédiction des émissions CO2 des voitures neuves en Europe (EEA)

**Contexte métier.** Agence européenne pour l'environnement et Commission européenne : les émissions moyennes CO2 des immatriculations neuves sont suivies constructeur par constructeur au titre du règlement (UE) 2019/631 (objectifs de réduction en 2025 puis 2030 vs 2021). Constructeurs, assureurs et journalistes veulent anticiper le CO2 WLTP d'un modèle avant sa mise sur le marché.

**Objectif de l'analyse exploratoire.** Identifier la cible (CO2 WLTP en g/km) et les régresseurs exploitables (masse, cylindrée, puissance, carburant, marque, pays d'immatriculation).

**Données.** EEA « Monitoring of CO2 emissions from passenger cars », table 2025 provisoire (10,8 millions d'immatriculations), interrogée via le point d'accès SQL DiscoData de l'EEA. Échantillon systématique 1/200 (ID % 200 = 0) pour l'EDA.

## 1. Téléchargement

La base complète fait 10,8 M de lignes : on requête un échantillon systématique (1 ligne sur 200) via le point d'accès SQL public DiscoData de l'EEA.

In [ ]:
import os, urllib.request

os.makedirs("data", exist_ok=True)
URL = ("https://discodata.eea.europa.eu/sql?query="
       "SELECT%20%5BID%5D%2C%5BMS%5D%2C%5BMk%5D%2C%5BCn%5D%2C%5BCt%5D%2C%5BFt%5D%2C%5BFm%5D%2C"
       "%5BEc%20(cm3)%5D%20AS%20ec_cm3%2C%5BEp%20(KW)%5D%20AS%20ep_kw%2C%5BM%20(kg)%5D%20AS%20m_kg%2C"
       "%5BMt%5D%20AS%20mt_kg%2C%5BEwltp%20(g/km)%5D%20AS%20ewltp_gkm%2C%5BZ%20(Wh/km)%5D%20AS%20z_whkm%2C"
       "%5BIT%5D%20AS%20innov_tech%20FROM%20%5BCO2Emission%5D.%5Blatest%5D.%5Bco2cars_2025Pv31%5D%20"
       "WHERE%20%5BID%5D%20%25%20200%20%3D%200")
if not os.path.exists("data/co2cars_sample.json"):
    urllib.request.urlretrieve(URL, "data/co2cars_sample.json")
print("OK", os.path.getsize("data/co2cars_sample.json") // 1024, "Ko")

## 2. Chargement

Le point d'accès renvoie du JSON ; on le convertit en DataFrame.

In [ ]:
import json
import pandas as pd

d = json.load(open("data/co2cars_sample.json"))["results"]
df = pd.DataFrame(d)
print(df.shape, "lignes x colonnes")
print(df.dtypes.to_string())
manq = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("\n% manquant par colonne :")
print(manq[manq > 0].to_string())

## 3. Analyse

Part des électriques et hybrides, distribution du CO2 WLTP par carburant, corrélations Spearman. Attention : la corrélation masse-CO2 ne se lit qu'à carburant constant.

In [ ]:
print("Part d'électriques :", round(100 * (df["Ft"] == "electric").mean(), 1), "%")
print("Part d'hybrides :", round(100 * df["Ft"].isin(["petrol/electric", "diesel/electric"]).mean(), 1), "%")
print()
print(df["Ft"].value_counts(normalize=True).mul(100).round(1).to_string())

therm = df[~df["Ft"].isin(["electric", "petrol/electric", "diesel/electric"])
           & df["ewltp_gkm"].notna() & df["ewltp_gkm"].gt(0)]
print()
print(therm["ewltp_gkm"].describe().round(1).to_string())
print()
print("CO2 médian par carburant :")
print(therm.groupby("Ft")["ewltp_gkm"].median().round(0).sort_values().to_string())
print()
print("Corrélation Spearman masse-CO2 par carburant :")
print(therm.groupby("Ft").apply(lambda g: g["m_kg"].corr(g["ewltp_gkm"], method="spearman")).round(2).to_string())
print()
num = therm.dropna(subset=["m_kg", "ep_kw", "ec_cm3"])
print("Corrélations Spearman (thermiques non hybrides) :")
print(num[["m_kg", "mt_kg", "ec_cm3", "ep_kw", "ewltp_gkm"]].corr(method="spearman")["ewltp_gkm"].round(2).to_string())

## 4. Résultats observés (exécution du 16/10/2026)

- base complète : 10 833 597 immatriculations (table 2025 provisoire) ; échantillon 0,5 % : 54 169 lignes, 14 colonnes
- part d'électriques : 18,8 % ; part d'hybrides : 9,9 %
- carburants : petrol 56,2 %, electric 18,8 %, diesel 11,7 %, petrol/electric 9,5 %, lpg 3,3 %
- CO2 WLTP (thermiques non hybrides, n = 38 574) : moyenne 131,8 g/km, médiane 126, 5 % : 101, 95 % : 190, max 442
- CO2 médian par carburant : ng 107, lpg 114, e85 123, petrol 124, diesel 143 g/km
- corrélations Spearman avec le CO2 : masse 0,57, masse en charge 0,59, cylindrée 0,44, puissance 0,71
- corrélation masse-CO2 à carburant constant : petrol 0,44, diesel 0,82 (globale trompeuse : 0,15 tous carburants confondus)
- valeurs manquantes : z_whkm 72,9 % (renseigné pour les électriques), ec_cm3 18,8 % (électriques), ep_kw 4,3 %, mt_kg 2,8 %
- top marques (échantillon) : TOYOTA 3 771, SKODA 3 687, RENAULT 3 524, VW 3 410, BMW 3 253 ; 29 pays
- CO2 médian par pays (top) : NO 170, IS 139, LU 136, DE 135 g/km

## 5. Variable cible et régresseurs

**Cible (régression) : `ewltp_gkm`** — émissions WLTP en g/km, sur les voitures thermiques non hybrides (les électriques ont ewltp = 0 par construction : les traiter à part, avec `z_whkm` en cible secondaire).

**Régresseurs retenus :**
- `m_kg` (masse à vide, corr. 0,57) et `ep_kw` (puissance, corr. 0,71) : les deux moteurs physiques de la cible
- `ec_cm3` (cylindrée), `mt_kg` (masse en charge)
- `Ft` (carburant : diesel ~ +19 g/km vs essence) et `Fm` (mode : hybride, classique)
- `Mk` (marque) et `MS` (pays d'immatriculation) : effets marché

**Pièges :**
- La corrélation masse-CO2 est nette à carburant constant (diesel 0,82) mais écrasée globalement (0,15) par les hybrides, lourds et sobres : toujours modéliser par carburant ou inclure `Ft` en feature
- `innov_tech` (technologies innovantes, éco-gains) décale la valeur homologuée vers le bas : feature informative mais déclarative
- Échantillon systématique 1/200 : reproductible mais ne pas extrapoler les variances sans re-pondérer par pays
- 2025 est une année provisoire (données révisées ensuite par l'EEA) ; l'électrique (18,8 %) doit être exclu de la cible CO2
- Ne pas utiliser `z_whkm` comme régresseur du CO2 : il ne concerne que les électriques (72,9 % de manquants ailleurs)